#    (PDF & TXT Summarization)
### part 1: Extractive Summarization by TF-IDF
### part 2: Abstractive Summarization by Transformer (BART)
### part 3: Comparision between two ways


In [1]:
# !pip uninstall -y transformers -q
# !pip install -q transformers==4.44.2 torch sentencepiece accelerate PyPDF2 spacy scikit-learn
# !python -m spacy download en_core_web_sm -q

In [2]:
import re
import time
import PyPDF2
import spacy
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline

nlp = spacy.load("en_core_web_sm")


## 1. read files  (PDF , TXT)


In [3]:
def read_txt_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    return text

def read_pdf_file(file_path):
    text = ""
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            text += page.extract_text() + " "
    return text

def read_file(file_path):
    """يحدد نوع الملف تلقائيًا ويقرأه (pdf أو txt)."""
    if file_path.lower().endswith('.pdf'):
        return read_pdf_file(file_path)
    return read_txt_file(file_path)


## 2.  Clean text


In [4]:

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    # Allows English (a-zA-Z), Arabic (\u0600-\u06FF), digits, and standard punctuation
    text = re.sub(r'[^a-zA-Z0-9\u0600-\u06FF\s.,!?]', '', text)
    text = text.strip()
    return text

def lemmatize_text(text):
    doc = nlp(text)
    lemmatized_tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(lemmatized_tokens)


## 3. Extractive Summarization by TF-IDF


In [5]:
def summarize_with_tfidf(text, num_sentences=15):
    if not text or not text.strip():
        return "Warning: Could not extract text from this file (it may be empty or a scanned PDF image)."

    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents if sent.text.strip()]

    if len(sentences) <= num_sentences:
        return text

    try:
        # (?u)\b\w+\b ensures Unicode characters (like Arabic) are recognized as words
        vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        # Fallback if text vocabulary is empty
        return " ".join(sentences[:num_sentences])

    sentence_scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
    ranked_sentences_indices = sentence_scores.argsort()[::-1]

    top_sentences_indices = ranked_sentences_indices[:num_sentences]
    top_sentences_indices.sort()

    summary_sentences = [sentences[i] for i in top_sentences_indices]
    return " ".join(summary_sentences)

## 4. Abstractive Summarization by Transformer

  Used model : **`facebook/bart-large-cnn`**


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# تحميل الموديل والتوكنير بشكل كامل
bart_model_name = "facebook/bart-large-cnn"
bart_tokenizer = AutoTokenizer.from_pretrained(bart_model_name)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(bart_model_name)

def chunk_text(text, tokenizer, max_tokens=1000):
    """تقسيم النص لقطع بناءً على التوكنز."""
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    input_ids = inputs["input_ids"][0]
    chunks = []
    for i in range(0, len(input_ids), max_tokens):
        chunk_ids = input_ids[i:i + max_tokens]
        chunks.append(tokenizer.decode(chunk_ids, skip_special_tokens=True))
    return chunks

def summarize_with_transformer(text, max_length=350, min_length=150):
    """تلخيص النص باستخدام BART مع الإعدادات الجديدة."""
    chunks = chunk_text(text, bart_tokenizer)
    summaries = []

    for chunk in chunks:
        inputs = bart_tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True)
        summary_ids = bart_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4,
            do_sample=False
        )
        summaries.append(bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True))

    combined_summary = " ".join(summaries)

    if len(summaries) > 1:
        inputs = bart_tokenizer(combined_summary, return_tensors="pt", max_length=1024, truncation=True)
        summary_ids = bart_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4
        )
        return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return combined_summary

c:\Users\moham\llm_env\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


 Used model : **`sshleifer/distilbart-cnn-12-6`**

In [7]:
distilbart_model_name = "sshleifer/distilbart-cnn-12-6"
distilbart_tokenizer = AutoTokenizer.from_pretrained(distilbart_model_name)
distilbart_model = AutoModelForSeq2SeqLM.from_pretrained(distilbart_model_name)

def summarize_with_distilbart(text, max_length=350, min_length=150):
    """تلخيص النص باستخدام DistilBART مع الإعدادات الجديدة."""
    chunks = chunk_text(text, distilbart_tokenizer)
    summaries = []

    for chunk in chunks:
        inputs = distilbart_tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True)
        summary_ids = distilbart_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4,
            do_sample=False
        )
        summaries.append(distilbart_tokenizer.decode(summary_ids[0], skip_special_tokens=True))

    combined_summary = " ".join(summaries)

    if len(summaries) > 1:
        inputs = distilbart_tokenizer(combined_summary, return_tensors="pt", max_length=1024, truncation=True)
        summary_ids = distilbart_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4
        )
        return distilbart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return combined_summary

 Used model (for arabic summarization) : **`fatmaserry/AraT5v2-arabic-summarization`**

In [8]:
arat5_summarizer = pipeline("summarization", model="fatmaserry/AraT5v2-arabic-summarization")

def summarize_with_arat5(text, max_length=350, min_length=150):
    """Summarizes Arabic text using the fine-tuned AraT5v2 model."""
    chunks = chunk_text(text)
    partial_summaries = []

    for chunk in chunks:
        # This specific AraT5 model accepts raw Arabic text directly without a "summarize: " prefix
        result = arat5_summarizer(chunk, max_length=max_length, min_length=min_length, do_sample=False)
        partial_summaries.append(result[0]['summary_text'])

    combined = " ".join(partial_summaries)

    # Re-summarize if the text was split into multiple chunks
    if len(chunks) > 1:
        final = arat5_summarizer(combined, max_length=max_length, min_length=min_length, do_sample=False)
        return final[0]['summary_text']

    return combined

## 5. Multilingual Summarization & Performance Comparison


In [15]:
from tkinter import Tk
from tkinter.filedialog import askopenfilename

root = Tk()
root.withdraw()
root.attributes("-topmost", True)

file_path = askopenfilename()

print(file_path)

root.destroy()

C:/Users/moham/OneDrive/Desktop/Story/Story 4.txt


In [16]:
# 2. Extract and Clean Text
raw_text = read_file(file_path)
clean = clean_text(raw_text)

# 3. Detect Language
def detect_language(text):
    """Detects if the text contains Arabic characters."""
    if re.search(r'[\u0600-\u06FF]', text):
        return "arabic"
    return "english"

language = detect_language(clean)

print("=" * 60)
print(f"Original text word count: {len(clean.split())}")
print(f"Detected Language: {language.upper()}")
print("=" * 60)

# Initialize container for model summaries
results = {}

Original text word count: 638
Detected Language: ENGLISH


In [17]:
# --- Extractive Baseline (TF-IDF) ---
start = time.time()
tfidf_summary = summarize_with_tfidf(clean, num_sentences=3)
tfidf_time = time.time() - start

print("--- TF-IDF (Extractive) ---")
print(f"Time: {tfidf_time:.2f}s | Word count: {len(tfidf_summary.split())}")
print(tfidf_summary)

results['tfidf'] = {"summary": tfidf_summary, "time": tfidf_time}

--- TF-IDF (Extractive) ---
Time: 0.39s | Word count: 97
Every night, when he turned off the lights, he felt something strange as if a pair of eyes were watching him from the far corner of the room, where no light reached. But over the days, he began to notice unexplained things a chair slightly out of place, a faint whisper of his name, and a shadow in the corner unmoving, yet present, like it was waiting. The paper inside was fragile, the handwriting slanted and soaked with regret I know you may never read this, but I write because I can no longer bear the silence.


In [18]:
# --- BART Model ---
if language == "english":
    start = time.time()
    transformer_summary = summarize_with_transformer(clean)
    transformer_time = time.time() - start

    print("--- Transformer / BART (Abstractive) ---")
    print(f"Time: {transformer_time:.2f}s | Word count: {len(transformer_summary.split())}")
    print(transformer_summary)

    results['transformer'] = {"summary": transformer_summary, "time": transformer_time}
else:
    print("Skipped BART: Detected language is ARABIC.")

--- Transformer / BART (Abstractive) ---
Time: 27.42s | Word count: 113
After a global soundbased disaster destroys civilization, one signal remains  a childs lullaby in Morse code. Clara, a former sound engineer, follows the signal with other survivors. It leads to a mountain facility where she discovers the truth the world wasnt destroyed it was paused. And now, its time to wake.,postapocalyptic, Morse code, survivors, haunting lullaby,Sound engineer, mysterious signal, global disaster,The world ended with sound. Years later, a signal returned. Repeating in Morse. Clara followed it with five others. Across ruins. Through snow. To the mountain. Inside cold cradles. A control panel. Lights flickered. The lullaby stopped. Clara closed her eyes and listened to the silence. It was beautiful. And full of beginnings.


In [19]:
# --- AraT5v2 Model ---
if language == "arabic":
    start = time.time()
    arat5_summary = summarize_with_arat5(clean)
    arat5_time = time.time() - start

    print("--- AraT5v2 (Abstractive) ---")
    print(f"Time: {arat5_time:.2f}s | Word count: {len(arat5_summary.split())}")
    print(arat5_summary)

    results['arat5'] = {"summary": arat5_summary, "time": arat5_time}
else:
    print("Skipped AraT5v2: Detected language is ENGLISH.")

Skipped AraT5v2: Detected language is ENGLISH.


In [20]:
# --- Final Summary Inspection ---
print("=" * 60)
print("ALL GENERATED RESULTS")
print("=" * 60)

for model_name, data in results.items():
    print(f"\n[{model_name.upper()}]")
    print(f"Time: {data['time']:.2f}s | Word Count: {len(data['summary'].split())}")
    print(data['summary'])

ALL GENERATED RESULTS

[TFIDF]
Time: 0.39s | Word Count: 97
Every night, when he turned off the lights, he felt something strange as if a pair of eyes were watching him from the far corner of the room, where no light reached. But over the days, he began to notice unexplained things a chair slightly out of place, a faint whisper of his name, and a shadow in the corner unmoving, yet present, like it was waiting. The paper inside was fragile, the handwriting slanted and soaked with regret I know you may never read this, but I write because I can no longer bear the silence.

[TRANSFORMER]
Time: 27.42s | Word Count: 113
After a global soundbased disaster destroys civilization, one signal remains  a childs lullaby in Morse code. Clara, a former sound engineer, follows the signal with other survivors. It leads to a mountain facility where she discovers the truth the world wasnt destroyed it was paused. And now, its time to wake.,postapocalyptic, Morse code, survivors, haunting lullaby,Sound e